# Train HRI-ACT on Google Colab

1. Locally: `./run.sh -m scripts.package_for_colab --dataset outputs/datasets/<dataset> --out outputs/colab/intent_policy.tar.gz`
2. Upload the archive to `MyDrive/intent_policy/`.
3. Runtime → GPU (T4). Run the cells. Checkpoints go to Drive; re-running the training cell after a disconnect resumes exactly.
4. Download `checkpoints/last/pretrained_model` and evaluate locally (`scripts/evaluate.py`, MuJoCo).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/intent_policy'
ARCHIVE = f'{DRIVE}/intent_policy.tar.gz'


In [ ]:
!tar -xzf "$ARCHIVE" -C /content
%cd /content/intent_policy
!pip -q install 'git+https://github.com/huggingface/lerobot.git@64b23178' pyav rerun-sdk==0.24.* 2>&1 | tail -2
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Settings: experiment config, run name, training seed and policy overrides (e.g. intent_mask=true for No-Information)
CONFIG = 'configs/experiments/phase2_oracle.yaml'
RUN = 'oracle_s0'
SEED = 0
POLICY_SET = 'intent_mask=false'
OUT = f'{DRIVE}/train/{RUN}'
import os; os.environ['PYTHONPATH'] = '/content/intent_policy'


In [ ]:
# Smoke test: 200 steps, measures data-loading vs GPU throughput (steps_per_s in the log)
!python -m scripts.train_policy --config $CONFIG --output-dir /content/smoke --steps 200 --num-workers 2 --video-backend pyav --policy-set $POLICY_SET 2>&1 | tail -3


In [ ]:
# Full training; re-run this cell after a disconnect: --resume continues from the newest checkpoint on Drive
!python -m scripts.train_policy --config $CONFIG --output-dir "$OUT" --seed $SEED --num-workers 2 --video-backend pyav --save-every 1000 --policy-set $POLICY_SET --resume 2>&1 | grep -E 'step=|saved|resumed|Error'
